# Lecture 2

### Motivation:

* What happens underneath when we call `eigh(...)`?
* How can we optimize our codes for performance by exploiting known properties of the matrices in question?
  
$\rightarrow$ explore Python documentation!
  
$\rightarrow$ Scipy modules use `lapack`, `arpack`, `blas` linear algebra libraries written in FORTRAN

In [20]:
import sys
sys.path.append("..")

import numpy as np
import qutip as qt
from qutip import Bloch
from scipy import sparse
from numpy.linalg import qr
from scipy.sparse import diags
from scipy.linalg import expm, eigh
from scipy.sparse import coo_matrix, csr_matrix
import Comp_Quant_Dynam as cqd   # custom package
from Comp_Quant_Dynam.operators import X, Y, Z, I
import quantum_algorithms as qa  # custom package 

# 2.1 Properties of Matrices (Operators)

Important classes of matrices (operators) include:

- Hermitian matrices
- Symmetric matrices
- Unitary matrices
- Normal matrices

These matrix classes are fundamental in linear algebra, quantum mechanics, and computational quantum dynamics.

## Hermitian matrices
A matrix H, which is $
H \in \mathbb{C}^{n \times n}
$
 $$ H^\dagger = (H^*)^T = H $$ it is called Hermitian. 
### Properties

- eigenvalues are real
- eigenvectors corresponding to different eigenvalues are orthogonal
- Hermitian matrices represent observables and Hamiltonians in quantum mechanics

In [21]:
# Hermitian matrix(operators) A  
A = np.array([[2, 1j], [-1j, 3]], dtype=complex)

# Check Hermitian property
print("A =\n", A)
print("A_dagger =\n", A.conj().T)
print("Is A Hermitian?", np.allclose(A, A.conj().T))

#eigenvalues and eigenvectors
eigenvalues, eigenvectors = eigh(A)
print("Eigenvalues=", eigenvalues)

# Check eigenvalues of Hermitian matrices is real
print("Are eigenvalues real?", np.all(np.isreal(eigenvalues)))

A =
 [[ 2.+0.j  0.+1.j]
 [-0.-1.j  3.+0.j]]
A_dagger =
 [[ 2.-0.j -0.+1.j]
 [ 0.-1.j  3.-0.j]]
Is A Hermitian? True
Eigenvalues= [1.38196601 3.61803399]
Are eigenvalues real? True


## Unitary matrices

A matrix H, which is $
H \in \mathbb{C}^{n \times n}$

 $$ \quad H^\dagger = H^{-1}=\quad H^\dagger H = H H^\dagger = \mathbb{1} $$ it is called Unitary 

 ### Properties
  * unit eigenvalues $|\lambda_i| = 1$
  * preserve angles and norm: $\langle \psi | \phi \rangle = \langle \psi | A^\dagger A | \phi \rangle$

e.g., time evolution operator $e^{-iHt}$, basis change

In [22]:
# unitary matrix
H = (1/np.sqrt(2)) * np.array([
    [1,  1],
    [-1, 1]
], dtype=complex)

# Conjugate transpose (Hermitian adjoint)
H_dagger = H.conj().T

# Inverse of A
H_inv = np.linalg.inv(H)

print("H:\n", H)
print("H† (conjugate transpose):\n", H_dagger)
print("H⁻¹ (inverse)=\n", H_inv)

# Check H† = H⁻¹
print("Check H† = H⁻¹, ", np.allclose(H_dagger, H_inv))

# Check H† H = I
print("H† H =\n", H_dagger @ H)
print("Check †H H = I, ", np.allclose(H_dagger @ H, I))

H:
 [[ 0.70710678+0.j  0.70710678+0.j]
 [-0.70710678+0.j  0.70710678+0.j]]
H† (conjugate transpose):
 [[ 0.70710678-0.j -0.70710678-0.j]
 [ 0.70710678-0.j  0.70710678-0.j]]
H⁻¹ (inverse)=
 [[ 0.70710678+0.j -0.70710678+0.j]
 [ 0.70710678+0.j  0.70710678+0.j]]
Check H† = H⁻¹,  True
H† H =
 [[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]
Check †H H = I,  True


## Normal matrices
A matrix H, which is $
H \in \mathbb{C}^{n \times n}
$
 $$ \quad H^\dagger H = H H^\dagger \quad $$ it is called Normal 

### Properties
  * all normal matrices are **diagonalizable**, i.e. there exists a unitary matrix $U$ such that $U^{-1}AU = D$ where $D$ is a diagonal matrix. $\quad (= U^\dagger A U)$

e.g., hermitian matrices, unitary matrices

In [23]:
#Normal matrix(A†A = AA†)
A = np.array([
    [1, -1],
    [1,  1]
], dtype=complex)
q, r = qr(A)
print("Q=\n", q)
print("R=\n", r)

# Hermitian conjugate (A†)
A_dagger = A.conj().T
print("A†A =\n", A_dagger @ A)
print("AA† =\n", A @ A_dagger)
print("Are they equal?,", np.allclose(A_dagger @ A, A @ A_dagger))

Q=
 [[-0.70710678+0.j -0.70710678+0.j]
 [-0.70710678-0.j  0.70710678+0.j]]
R=
 [[-1.41421356e+00+0.j -3.33066907e-16+0.j]
 [ 0.00000000e+00+0.j  1.41421356e+00+0.j]]
A†A =
 [[2.+0.j 0.+0.j]
 [0.+0.j 2.+0.j]]
AA† =
 [[2.+0.j 0.+0.j]
 [0.+0.j 2.+0.j]]
Are they equal?, True


## Symmetric matrix

A matrix H, which is $
H \in \mathbb{C}^{n \times n}
$
 $$ \quad H = H^T\quad $$ it is called Symmetric.

### Properties

- entries satisfy

$$
H_{ij} = H_{ji}
$$

- if all entries are real, the matrix is symmetric about the main diagonal

- symmetric matrices are a special case of Hermitian matrices when all entries are real

- eigenvalues of real symmetric matrices are real

- eigenvectors corresponding to different eigenvalues are orthogonal

example. 
- Symmetric: \(I, X, Z, S, T\)
- Not symmetric: \(Y\)

In [24]:
#Symmetric
X_tran = X.T
print("X_tran =\n", X_tran)
print("Are they equal?,", np.allclose(X, X_tran))

X_tran =
 [[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]
Are they equal?, True


In [25]:
#Symmetric
Y_tran = Y.T
print("Y_tran =\n", Y_tran)
print("Are they equal?,", np.allclose(Y, Y_tran))

Y_tran =
 [[ 0.+0.j  0.+1.j]
 [-0.-1.j  0.+0.j]]
Are they equal?, False


#### Idea behind most numerical diagonalization algorithms

 Find a series of unitary transformations 

$$A \rightarrow U_1^{-1} A U_1 \rightarrow U_2^{-1} U_1^{-1} A U_1 U_2 \rightarrow \dots \rightarrow D$$

that converges to a diagonal matrix $\quad \underbrace{U_1 \dots U_n}_{:= U} = U$

Then, approximate eigenvectors will be the columns of $U$.

# 2.2 QR Algorithm (QR Iteration)

The QR algorithm is a numerical linear algebra eigenvalue algorithm; that is, it is a procedure used to calculate the eigenvalues and eigenvectors of a matrix.

The basic idea is to perform a QR decomposition by writing the matrix as a product of an **orthogonal (or unitary) matrix** and an **upper triangular matrix**. The factors are then multiplied in reverse order, and this process is repeated iteratively.

The QR algorithm is one of the most important numerical methods for computing **eigenvalues** and **eigenvectors**.

## QR Decomposition

A = QR


where:

- \(Q\) is a unitary (or orthogonal) matrix
- \(R\) is an upper triangular matrix

The QR algorithm iteratively computes QR decompositions:

$$
\left.
\begin{aligned}
A_0 &= Q_1 R_1 \\
A_1 &= R_1 Q_1
\end{aligned}
\right\}
\text{iterate}
$$

$$
\left.
\begin{aligned}
A_1 &= Q_2 R_2 \\
A_2 &= R_2 Q_2
\end{aligned}
\right\}
\text{iterate}
$$

$$
\left.
\begin{aligned}
A_2 &= Q_3 R_3 \\
A_3 &= R_3 Q_3
\end{aligned}
\right\}
\text{iterate}
$$

$$
\left.
\begin{aligned}
A_3 &= Q_4 R_4 \\
A_4 &= R_4 Q_4
\end{aligned}
\right\}
\text{iterate}
$$

$$
\left.
\begin{aligned}
A_4 &= Q_5 R_5 \\
A_5 &= R_5 Q_5
\end{aligned}
\right\}
\text{iterate}
$$

$$
\vdots
$$

with

$$
\begin{aligned}
Q &: \text{unitary matrix} \\
R &: \text{upper triangular matrix}
\end{aligned}
$$

An algorithm for computing \(Q\) and \(R\) from a given matrix \(A\) is known.

Using

$$
R_n = Q_n^{-1} A_{n-1},
$$

one obtains

$$
A_n = R_n Q_n = Q_n^{-1} A_{n-1} Q_n.
$$

Therefore, each iteration is a similarity transformation.

The sequence

Series $A_0, A_1, A_2 \dots$ **converges to a diagonal matrix** $D$  
off-diagonal elements decay as

$$a_{ij}^{(k)} \sim \left| \frac{\lambda_i}{\lambda_j} \right|^k \qquad \begin{aligned} &\text{where wlog } i < j \text{ and} \\ &\text{sorted eigenvalues } \lambda_1 \gg \lambda_2 \gg \dots \\ &\text{are assumed.} \end{aligned}$$

Hence, the QR algorithm produces an approximate eigenvalue decomposition:

$$
D \approx A_n
=
R_n Q_n
=
Q_n^{-1} A_{n-1} Q_n
=
\cdots
=
\underbrace{Q_n^{-1}\cdots Q_1^{-1}}_{U^\dagger}
A
\underbrace{Q_1 \cdots Q_n}_{U}.
$$

The columns of \(U\) approximate the eigenvectors of \(A\), while the diagonal entries of \(D\) approximate the eigenvalues.



# Problem:
* **Convergence can be slow** if the eigenvalues are closely spaced.
* **Computationally costly:** each QR decomposition step requires $\mathcal{O}(N^3)$ operations. Since a single QR iteration has a cost of $\mathcal{O}(N^3)$ and the convergence is linear, the standard QR algorithm can become computationally very expensive.

### Solution: Add a reduction ("preconditioning") step

The total number of reduction steps is generally **$(N-2)$**.

1. Bring $A$ to Hessenberg form
   $$
   \begin{pmatrix}
   \dots & \dots & \dots \\
   \dots & \dots & \dots \\
   0 & \dots & \dots
   \end{pmatrix}
   $$
   using **<u>Householder transformations</u>**.  
   *(For Hermitian $A$, this reduces to a tridiagonal form.)*

2. Apply the QR iteration using **<u>Givens rotations</u>**.

### Householder transformation

Householder transformations are used to reduce a matrix to Hessenberg form.

For a Hermitian matrix, the Hessenberg form becomes tridiagonal.

A Householder reflector is defined by

$$
P = \mathbb{1} - 2|u\rangle\langle u|
$$

where \(u\) is a normalized vector:

$$
\|u\| = 1
$$

The reflector is unitary and Hermitian:

$$
P^\dagger = P^{-1} = P
$$

Householder transformations are used to successively eliminate matrix elements below the first subdiagonal.

In [30]:
# Original 3x3 matrix
A = np.array([
    [-1, 0, 10],
    [6, 5, 2],
    [3, 33, 7],
], dtype=float)

q, r = qr(A)
print("Q=\n", q)
print('R=\n', r)

# Matrix size
n = A.shape[0]
#Construct Householder Reflection P1
V = A[1:, 0]#(Take first column below diagonal)
print("Vector(V)=", V)

# Compute norm of V
norm_V = np.linalg.norm(V)

# Choose alpha(we can choose any we want )
alpha = -np.sign(V[0]) * norm_V
print("alpha = ", alpha)

# First basis vector e1
e1 = np.zeros_like(V)
e1[0] = 1

# Construct Householder vector u
u = V - alpha * e1

# Normalize u(Normalized Householder vector u)
u = u / np.linalg.norm(u)
print("u=", u)

# Construct a small Householder matrix, P_small = I - 2uu^T
P1_small = np.eye(len(V)) - 2 * np.outer(u, u)
print("P_small:\n", P1_small)

# Embed into full matrix P1
P1 = np.eye(n)

# Insert P_small into lower-right block
P1[1:, 1:] = P1_small
print("P1:\n", P1) #full Householder matrix P1

# Transformed matrix A1,  A1 = P1^T A P1(from A to A1 by Embed into full matrix P1)
A1 = P1.T @ A @ P1
print("A1:\n", A1)

#second track 
#Construct Householder Reflection P2
# Now work on the lower-right 3x3 block of A1
# Take column 2 below the diagonal

V2 = A1[2:, 1] #(Take second column below diagonal)
print("V2=\n", V2) #Vector V2 (entries below a22)
norm_V2 = np.linalg.norm(V2)

# Choose alpha
alpha2 = -np.sign(V2[0]) * norm_V2
print("alpha2 = ", alpha2)

# Basis vector e1
e1_2 = np.zeros_like(V2)
e1_2[0] = 1

# Construct Householder vector
u2 = V2 - alpha2 * e1_2
u2 = u2 / np.linalg.norm(u2)
print("u2:\n", u2)

# Small Householder matrix
P2_small = np.eye(len(V2)) - 2 * np.outer(u2, u2)
print("P2_small:\n", P2_small)
# Embed into full 4x4 matrix
P2 = np.eye(n)

# Insert into lower-right corner
P2[2:, 2:] = P2_small
print("P2:\n", P2)

# Second similarity transformation
A2 = P2.T @ A1 @ P2
print("A2:\n", A2)

Q=
 [[-0.14744196  0.10224779  0.98377145]
 [ 0.88465174 -0.43118447  0.17740141]
 [ 0.44232587  0.89645153 -0.026879  ]]
R=
 [[ 6.78232998 19.02001234  3.39116499]
 [ 0.         27.42697815  6.43526965]
 [ 0.          0.         10.00436428]]
Vector(V)= [6. 3.]
alpha =  -6.708203932499369
u= [0.97324899 0.22975292]
P_small:
 [[-0.89442719 -0.4472136 ]
 [-0.4472136   0.89442719]]
P1:
 [[ 1.          0.          0.        ]
 [ 0.         -0.89442719 -0.4472136 ]
 [ 0.         -0.4472136   0.89442719]]
A1:
 [[-1.00000000e+00 -4.47213595e+00  8.94427191e+00]
 [-6.70820393e+00  1.94000000e+01  4.20000000e+00]
 [ 6.66133815e-16 -2.68000000e+01 -7.40000000e+00]]
V2=
 [-26.8]
alpha2 =  26.799999999999994
u2:
 [-1.]
P2_small:
 [[-1.]]
P2:
 [[ 1.  0.  0.]
 [ 0.  1.  0.]
 [ 0.  0. -1.]]
A2:
 [[-1.00000000e+00 -4.47213595e+00 -8.94427191e+00]
 [-6.70820393e+00  1.94000000e+01 -4.20000000e+00]
 [-6.66133815e-16  2.68000000e+01 -7.40000000e+00]]


In [31]:
# Original 4x4 matrix
A = np.array([
    [9, 13, 3, 2],
    [1, 11, 4, 1],
    [3, 7, 4, 1],
    [6, 0, 7, 10]
], dtype=float)

q, r = qr(A)
print("Q=\n", q)
print('R=\n', r)

# Matrix size
n = A.shape[0]
#Construct Householder Reflection P1
V = A[1:, 0]#(Take first column below diagonal)
print("Vector(V)=", V)

# Compute norm of V
norm_V = np.linalg.norm(V)

# Choose alpha(we can choose any we want )
alpha = -np.sign(V[0]) * norm_V
print("alpha = ", alpha)

# First basis vector e1
e1 = np.zeros_like(V)
e1[0] = 1

# Construct Householder vector u
u = V - alpha * e1

# Normalize u(Normalized Householder vector u)
u = u / np.linalg.norm(u)
print("u=", u)

# Construct a small Householder matrix, P_small = I - 2uu^T
P1_small = np.eye(len(V)) - 2 * np.outer(u, u)
print("P_small:\n", P1_small)

# Embed into full matrix P1
P1 = np.eye(n)

# Insert P_small into lower-right block
P1[1:, 1:] = P1_small
print("P1:\n", P1) #full Householder matrix P1

# Transformed matrix A1,  A1 = P1^T A P1(from A to A1 by Embed into full matrix P1)
A1 = P1.T @ A @ P1
print("A1:\n", A1)

#second track 
#Construct Householder Reflection P2
# Now work on the lower-right 3x3 block of A1
# Take column 2 below the diagonal

V2 = A1[2:, 1] #(Take second column below diagonal)
print("V2=\n", V2) #Vector V2 (entries below a22)
norm_V2 = np.linalg.norm(V2)

# Choose alpha
alpha2 = -np.sign(V2[0]) * norm_V2
print("alpha2 = ", alpha2)

# Basis vector e1
e1_2 = np.zeros_like(V2)
e1_2[0] = 1

# Construct Householder vector
u2 = V2 - alpha2 * e1_2
u2 = u2 / np.linalg.norm(u2)
print("u2:\n", u2)

# Small Householder matrix
P2_small = np.eye(len(V2)) - 2 * np.outer(u2, u2)
print("P2_small:\n", P2_small)
# Embed into full 4x4 matrix
P2 = np.eye(n)

# Insert into lower-right corner
P2[2:, 2:] = P2_small
print("P2:\n", P2)

# Second similarity transformation
A2 = P2.T @ A1 @ P2
print("A2:\n", A2)

Q=
 [[-0.79862086 -0.19049605  0.56120143  0.10473267]
 [-0.08873565 -0.76690023 -0.4670494   0.4311089 ]
 [-0.26620695 -0.2716105  -0.30829646 -0.87196037]
 [-0.53241391  0.54936603 -0.60981235  0.2070297 ]]
R=
 [[-11.26942767 -13.22161199  -7.54253033  -7.27632338]
 [  0.         -12.81362464  -0.87996885   4.07415746]
 [  0.           0.          -5.6864656   -5.75106651]
 [  0.           0.           0.           1.83891084]]
Vector(V)= [1. 3. 6.]
alpha =  -6.782329983125268
u= [0.75744371 0.29198597 0.58397193]
P_small:
 [[-0.14744196 -0.44232587 -0.88465174]
 [-0.44232587  0.82948839 -0.34102322]
 [-0.88465174 -0.34102322  0.31795356]]
P1:
 [[ 1.          0.          0.          0.        ]
 [ 0.         -0.14744196 -0.44232587 -0.88465174]
 [ 0.         -0.44232587  0.82948839 -0.34102322]
 [ 0.         -0.88465174 -0.34102322  0.31795356]]
A1:
 [[ 9.00000000e+00 -5.01302651e+00 -3.94381755e+00 -1.18876351e+01]
 [-6.78232998e+00  1.28260870e+01 -1.78853182e+00  4.08991808e+00]
 

In [32]:
# Original 4x4 matrix
A = np.array([
    [37, 600, 13, 749, 2],
    [43, 11, 9,  4, 198],
    [3, 8888, 1000, 4, 400000],
    [777, 0, 68, 16, 10],
    [2, 5, 18, 66, 120]
], dtype=float)

q, r = qr(A)
print("Q=\n", q)
print('R=\n', r)

# Matrix size
n = A.shape[0]
#Construct Householder Reflection P1
V = A[1:, 0]#(Take first column below diagonal)
print("Vector(V)=", V)

# Compute norm of V
norm_V = np.linalg.norm(V)

# Choose alpha(we can choose any we want )
alpha = -np.sign(V[0]) * norm_V
print("alpha = ", alpha)

# First basis vector e1
e1 = np.zeros_like(V)
e1[0] = 1

# Construct Householder vector u
u = V - alpha * e1

# Normalize u(Normalized Householder vector u)
u = u / np.linalg.norm(u)
print("u=", u)

# Construct a small Householder matrix, P_small = I - 2uu^T
P1_small = np.eye(len(V)) - 2 * np.outer(u, u)
print("P_small:\n", P1_small)

# Embed into full matrix P1
P1 = np.eye(n)

# Insert P_small into lower-right block
P1[1:, 1:] = P1_small
print("P1:\n", P1) #full Householder matrix P1

# Transformed matrix A1,  A1 = P1^T A P1(from A to A1 by Embed into full matrix P1)
A1 = P1.T @ A @ P1
print("A1:\n", A1)

#second track 
#Construct Householder Reflection P2
# Now work on the lower-right 4x4 block of A1
# Take column 2 below the diagonal

V2 = A1[2:, 1] #(Take second column below diagonal)
print("V2=\n", V2) #Vector V2 (entries below a22)
norm_V2 = np.linalg.norm(V2)

# Choose alpha
alpha2 = -np.sign(V2[0]) * norm_V2
print("alpha2 = ", alpha2)

# Basis vector e1
e1_2 = np.zeros_like(V2)
e1_2[0] = 1

# Construct Householder vector
u2 = V2 - alpha2 * e1_2
u2 = u2 / np.linalg.norm(u2)
print("u2:\n", u2)

# Small Householder matrix
P2_small = np.eye(len(V2)) - 2 * np.outer(u2, u2)
print("P2_small:\n", P2_small)
# Embed into full 4x4 matrix
P2 = np.eye(n)

# Insert into lower-right corner
P2[2:, 2:] = P2_small
print("P2:\n", P2)

# Second similarity transformation
A2 = P2.T @ A1 @ P2
print("A2:\n", A2)

#Third track (5-2=3) three steps to 
#Construct Householder Reflection P3
# Now work on the lower-right 4x4 block of A3
# Take column 4 below the diagonal

V3 = A2[3:, 2] #(Take second column below diagonal)
print("V3=\n", V3) #Vector V2 (entries below a22)
norm_V3 = np.linalg.norm(V3)

# Choose alpha
alpha3 = -np.sign(V3[0]) * norm_V3
print("alpha3 = ", alpha3)

# Basis vector e1
e1_3 = np.zeros_like(V3)
e1_3[0] = 1

# Construct Householder vector
u3 = V3 - alpha3 * e1_3
u3 = u3 / np.linalg.norm(u3)
print("u3:\n", u3)

# Small Householder matrix
P3_small = np.eye(len(V3)) - 2 * np.outer(u3, u3)
print("P3_small:\n", P3_small)
# Embed into full 5x5 matrix
P3 = np.eye(n)

# Insert into lower-right corner
P3[3:, 3:] = P3_small
print("P3:\n", P3)

# third similarity transformation
A3 = P3.T @ A2 @ P3
print("A3:\n", A3)

Q=
 [[-4.74921344e-02  6.70174029e-02  9.52495758e-01  2.93006866e-01
  -1.23276845e-02]
 [-5.51935616e-02  8.42389792e-04 -6.87485543e-02  1.73074764e-01
  -9.80954496e-01]
 [-3.85071360e-03  9.97726101e-01 -6.40534454e-02 -2.05227966e-02
   1.94159106e-03]
 [-9.97334823e-01 -7.09154575e-03 -4.05666868e-02 -2.58704893e-02
   5.43877006e-02]
 [-2.56714240e-03  5.43038806e-04 -2.86839618e-01  9.39734875e-01
   1.86049759e-01]]
R=
 [[-7.79076376e+02 -6.33403881e+01 -7.28298300e+01 -5.19345744e+01
  -1.56159016e+03]
 [ 0.00000000e+00  8.90801201e+03  9.98132459e+02  5.41126845e+01
   3.99090736e+05]
 [ 0.00000000e+00  0.00000000e+00 -6.02113854e+01  6.93307633e+02
  -2.56679118e+04]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  2.81680924e+02
  -8.06175434e+03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   6.05252627e+02]]
Vector(V)= [ 43.   3. 777.   2.]
alpha =  -778.1972757598165
u= [0.72638004 0.00265361 0.68728587 0.00176908]
P_small:
 [[-5.52559118e-02 -3

# Hessenberg Matrix Structure in the QR Algorithm

For the (4 by 4) Hessenberg matrix above, the QR algorithm mainly operates on the nonzero entries.

## Approximately Zero Entries

The following entries are approximately zero:

$$
a_{31}, \quad a_{41}, \quad a_{42}
$$

These values are only floating-point numerical roundoff errors.
Computationally, they are treated as zero during QR iterations.

### Computational Advantages of Hessenberg Reduction

The QR algorithm becomes more efficient because:

- most lower-left entries are zero
- fewer operations are required
- fewer matrix elements are updated
- cache locality improves
- memory movement decreases

This is exactly why Hessenberg preprocessing is performed before QR iterations.

### Cache Locality

Cache locality means that data used during computations is stored close together in memory, allowing the CPU to access the data more efficiently.

Modern processors use small high-speed memory called **cache** because accessing RAM is much slower than performing arithmetic operations.

When the Hessenberg matrix contains mostly zeros, the QR algorithm mainly accesses nearby diagonal elements:

$$
\begin{pmatrix}
* & * & * & * \\
* & * & * & * \\
0 & * & * & * \\
0 & 0 & * & *
\end{pmatrix}
$$

Therefore:

- fewer matrix entries are accessed
- nearby memory locations are reused
- fewer cache misses occur
- computations become faster

This is called **better cache locality**.

In [29]:
a = np.array([[0, 2], 
              [2, 3]])
p = [1, 5, 10, 20]
for i in range(20):
    q, r = qr(a)
    a = np.dot(r, q)
    if i+1 in p:
        print(f'Iteration {i+1}:')
        print(a)

Iteration 1:
[[3. 2.]
 [2. 0.]]
Iteration 5:
[[ 3.99998093  0.00976559]
 [ 0.00976559 -0.99998093]]
Iteration 10:
[[ 4.00000000e+00  9.53674316e-06]
 [ 9.53674316e-06 -1.00000000e+00]]
Iteration 20:
[[ 4.00000000e+00  9.09484251e-12]
 [ 9.09494702e-12 -1.00000000e+00]]


### Matrix Transformations & Subspace Operators

| Left Multiplication Vector Mapping | Output Structural Representation | Diagonal Block Expansion ($\mathbb{P}$) |
| :---: | :---: | :---: |
| Applying operator $P$ from the left side transforms the initial tracking column to: | $\begin{pmatrix} a_{11} \\ \alpha_{21} \\ 0 \\ \vdots \end{pmatrix}$ <br><br> *(note: running the step from the right side leaves column 1 completely unaffected)* | Re-run the structural sequence down the subspace hierarchy: <br><br> $\mathbb{P} = \begin{pmatrix} 1 & 0 & 0 & \dots \\ 0 & 1 & 0 & \dots \\ 0 & 0 & & \\ \vdots & \vdots & & \mathbb{P} \end{pmatrix}$ |

<br>

**Overall Algorithmic Complexity Scaling:** $\quad \frac{4}{3} N^3$

### Complexity Theory and QR Algorithms

This connects directly to **complexity theory** and **computational efficiency** in numerical linear algebra.

Algorithms are mainly evaluated by:

- **Time complexity**  
  (operations take more time to evaluate and calculate the computations based on the size of the data received. Sometimes systems may become slow because of long operational times and heavy computations.)

- **Space complexity (memory)**  
  (measures how much memory/storage is required for computations and data processing. Large computations usually require large memory resources.)

- **Energy / hardware cost**  
  (large computational systems require powerful hardware, massive data centers, cooling systems, and optimization infrastructure. For example, current AI systems require huge data centers and advanced cooling systems in order to operate efficiently.)

  
  
# Dense QR vs Hessenberg QR

Suppose \(A\) is a dense matrix.

A standard QR iteration on a dense matrix costs:

$$
\mathcal{O}(N^3)
$$

because every QR step manipulates almost all matrix entries.  
A dense matrix has almost all nonzero elements, therefore all these values must be stored and processed, which requires:

- larger memory usage
- more computational operations
- more memory movement between hardware

This increases both computational time and hardware cost.

# After Hessenberg Reduction

After Hessenberg reduction, most entries become zero:

$$
H =
\begin{pmatrix}
* & * & * & * \\
* & * & * & * \\
0 & * & * & * \\
0 & 0 & * & *
\end{pmatrix}
$$

Most elements are now zero, so less memory is required because zero entries do not need heavy computation or storage.

QR iterations now only affect nearby diagonals instead of the entire matrix.

Therefore, the computational cost per QR sweep becomes:

$$
\mathcal{O}(N^2)
$$

This significantly reduces:

- runtime
- memory usage
- energy consumption

which is why Hessenberg reduction is an important preprocessing step in numerical linear algebra.

# 2.3 Sparse matrices
In many numerical algorithms such as the QR algorithm, solving matrix problems using dense matrices can require large amounts of computational time and memory.
To reduce these costs, we often use sparse matrices, which store only the important non-zero elements and ignore unnecessary zeros.

This is especially important in complexity theory and scientific computing, where reducing memory usage and computational operations improves efficiency.

The Hamiltonian matrix H of many physical systems is often sparse, meaning that most of its elements are zero.

Typically: **$\text{\# non-zero elements} \propto N$**

## Sparse matrix or sparse array
It is a matrix in which most of the elements are ***zero***. There is no strict definition regarding the proportion of zero-value elements for a matrix to qualify as sparse but ***a common criterion is that the number of non-zero elements is roughly equal to the number of rows or columns***. By contrast, if most of the elements are non-zero, the matrix is considered dense. The number of zero-valued elements divided by the total number of elements 

* **Example:** (m × n for an m × n matrix) is sometimes referred to as the sparsity of the matrix. 

### Applications of Sparse matrices

When storing and manipulating sparse matrices on a quantum computer, it is beneficial and often necessary to use specialized algorithms and data structures that take advantage of the sparse structure of the matrix. Specialized quantum computers have been made for sparse matrices.  

Operations using standard dense-matrix structures and algorithms are ***slow, inefficient***. when applied to large sparse matrices, as processing and memory are wasted on the zeros. Sparse data is by nature more easily compressed and thus requires significantly *less storage* used to be small memory for CPU processors. Some very large sparse matrices are infeasible to manipulate using standard dense-matrix algorithms.


* **Example:**
The finite difference Hamiltonian was tri-diagonal (cf. lecture 1), which means there are only $\sim 3N$ non-zero elements.

$$\implies \text{It would be wasteful to store all the zeros!}$$



### Ways to store sparse matrices:

* **Index-value pairs:** $M = \{(i_1, j_1, v_1), (i_2, j_2, v_2), \dots\}$
  * Stores: `row index`, `col index`, `value of non-zero element`.
  * **`scipy` implementation:** `coo_matrix`

* **Compressed Format Structure:** $$M = \Big\{ \big\{ (j_{11}, v_{11}), (j_{12}, v_{12}), \dots \big\} \leftarrow \text{row 1}, \ \big\{ (j_{21}, v_{21}), \dots \big\} \leftarrow \text{row 2}, \ \dots \Big\}$$
  * Stores the column index and value of each non-zero element within its respective row (e.g., the column index and value of the 1st non-zero element in row 2).
  * **`scipy` implementation:** `csr` (see also `csc`)

* **Banded matrices:** Store the $k$-th off-diagonals as vectors:
  * $k = 0$ is the main diagonal ($N$-component vector)
  * $k = \pm 1$ represents the first off-diagonals ($(N-1)$-component vector), etc.
  * **`scipy` implementation:** `dia`


### Important Numerical Warnings & Performance:

| Matrix Storage Rules | Computational Best Practices | Efficient Solver Algorithm |
| :---: | :---: | :---: |
| ⚠️ **Don't use sparse matrices as input to numpy functions!** | NumPy functions expect contiguous arrays and can accidentally explode the matrix back into memory-heavy dense formats. | There are very efficient methods for finding a **few eigen-pairs** of a sparse matrix $\implies$ **Lanczos algorithm**. |

In [9]:
row = np.array([0,1,2])
col = np.array([0,1,2])
data = np.array([10,20,30])

A = coo_matrix((data, (row, col)), shape=(3,3))

print(A)
print(A.toarray())

  (0, 0)	10
  (1, 1)	20
  (2, 2)	30
[[10  0  0]
 [ 0 20  0]
 [ 0  0 30]]


In [10]:
# Dense matrix
A = np.array([
    [10,0,0,0],
    [3,9,0,0],
    [0,7,8,0],
    [0,0,2,6]
])

# Convert to sparse COO format
A_coo = sparse.coo_matrix(A)
A_csr = sparse.csr_matrix(A)
print("COO sparse matrix=\n", A_coo)
print("CSR sparse matrix=\n", A_csr)
print("H1 == H2 :", (A_coo != A_csr).nnz == 0)

COO sparse matrix=
   (0, 0)	10
  (1, 0)	3
  (1, 1)	9
  (2, 1)	7
  (2, 2)	8
  (3, 2)	2
  (3, 3)	6
CSR sparse matrix=
   (0, 0)	10
  (1, 0)	3
  (1, 1)	9
  (2, 1)	7
  (2, 2)	8
  (3, 2)	2
  (3, 3)	6
H1 == H2 : True


In [12]:
N = 8

main_diag = 2*np.ones(N)
off_diag  = -1*np.ones(N-1)

H1 = diags(
    [off_diag, main_diag, off_diag],
    offsets=[-1,0,1],
    format='csr'
)

H2 = diags(
    [main_diag, off_diag, off_diag],
    offsets=[0,1,-1],
    format='coo'
)

print("H=\n", H1)
print("H=\n", H2)
print("H1 == H2 :", (H1 != H2).nnz == 0)

H=
   (0, 0)	2.0
  (0, 1)	-1.0
  (1, 0)	-1.0
  (1, 1)	2.0
  (1, 2)	-1.0
  (2, 1)	-1.0
  (2, 2)	2.0
  (2, 3)	-1.0
  (3, 2)	-1.0
  (3, 3)	2.0
  (3, 4)	-1.0
  (4, 3)	-1.0
  (4, 4)	2.0
  (4, 5)	-1.0
  (5, 4)	-1.0
  (5, 5)	2.0
  (5, 6)	-1.0
  (6, 5)	-1.0
  (6, 6)	2.0
  (6, 7)	-1.0
  (7, 6)	-1.0
  (7, 7)	2.0
H=
   (0, 0)	2.0
  (1, 1)	2.0
  (2, 2)	2.0
  (3, 3)	2.0
  (4, 4)	2.0
  (5, 5)	2.0
  (6, 6)	2.0
  (7, 7)	2.0
  (0, 1)	-1.0
  (1, 2)	-1.0
  (2, 3)	-1.0
  (3, 4)	-1.0
  (4, 5)	-1.0
  (5, 6)	-1.0
  (6, 7)	-1.0
  (1, 0)	-1.0
  (2, 1)	-1.0
  (3, 2)	-1.0
  (4, 3)	-1.0
  (5, 4)	-1.0
  (6, 5)	-1.0
  (7, 6)	-1.0
H1 == H2 : True


In [13]:
N = 8

main_diag = 2*np.ones(N)
off_diag  = -1*np.ones(N-1)

H1 = diags(
    [off_diag, main_diag, off_diag],
    offsets=[-1,0,1],
    format='csr'
)

H2 = diags(
    [main_diag, off_diag, off_diag],
    offsets=[0,1,-1],
    format='csr'
)

print("H=\n", H1)
print("H=\n", H2)
print("H1 == H2 :", (H1 != H2).nnz == 0)

H=
   (0, 0)	2.0
  (0, 1)	-1.0
  (1, 0)	-1.0
  (1, 1)	2.0
  (1, 2)	-1.0
  (2, 1)	-1.0
  (2, 2)	2.0
  (2, 3)	-1.0
  (3, 2)	-1.0
  (3, 3)	2.0
  (3, 4)	-1.0
  (4, 3)	-1.0
  (4, 4)	2.0
  (4, 5)	-1.0
  (5, 4)	-1.0
  (5, 5)	2.0
  (5, 6)	-1.0
  (6, 5)	-1.0
  (6, 6)	2.0
  (6, 7)	-1.0
  (7, 6)	-1.0
  (7, 7)	2.0
H=
   (0, 0)	2.0
  (0, 1)	-1.0
  (1, 0)	-1.0
  (1, 1)	2.0
  (1, 2)	-1.0
  (2, 1)	-1.0
  (2, 2)	2.0
  (2, 3)	-1.0
  (3, 2)	-1.0
  (3, 3)	2.0
  (3, 4)	-1.0
  (4, 3)	-1.0
  (4, 4)	2.0
  (4, 5)	-1.0
  (5, 4)	-1.0
  (5, 5)	2.0
  (5, 6)	-1.0
  (6, 5)	-1.0
  (6, 6)	2.0
  (6, 7)	-1.0
  (7, 6)	-1.0
  (7, 7)	2.0
H1 == H2 : True


In [14]:
N = 20

main_diag = 2*np.ones(N)
off_diag  = -1*np.ones(N-1)

#In sparse diagonal matrix construction, the main diagonal always corresponds to offset 0.
H1 = diags(
    [off_diag, main_diag, off_diag],
    offsets=[-1,0,1],
    format='csr'
)

H2 = diags(
    [main_diag, off_diag, off_diag],
    offsets=[0,1,-1],
    format='csr'
)

print("H=\n", H1)
print("H=\n", H2)
print("H1 == H2 :", (H1 != H2).nnz == 0)

H=
   (0, 0)	2.0
  (0, 1)	-1.0
  (1, 0)	-1.0
  (1, 1)	2.0
  (1, 2)	-1.0
  (2, 1)	-1.0
  (2, 2)	2.0
  (2, 3)	-1.0
  (3, 2)	-1.0
  (3, 3)	2.0
  (3, 4)	-1.0
  (4, 3)	-1.0
  (4, 4)	2.0
  (4, 5)	-1.0
  (5, 4)	-1.0
  (5, 5)	2.0
  (5, 6)	-1.0
  (6, 5)	-1.0
  (6, 6)	2.0
  (6, 7)	-1.0
  (7, 6)	-1.0
  (7, 7)	2.0
  (7, 8)	-1.0
  (8, 7)	-1.0
  (8, 8)	2.0
  :	:
  (11, 11)	2.0
  (11, 12)	-1.0
  (12, 11)	-1.0
  (12, 12)	2.0
  (12, 13)	-1.0
  (13, 12)	-1.0
  (13, 13)	2.0
  (13, 14)	-1.0
  (14, 13)	-1.0
  (14, 14)	2.0
  (14, 15)	-1.0
  (15, 14)	-1.0
  (15, 15)	2.0
  (15, 16)	-1.0
  (16, 15)	-1.0
  (16, 16)	2.0
  (16, 17)	-1.0
  (17, 16)	-1.0
  (17, 17)	2.0
  (17, 18)	-1.0
  (18, 17)	-1.0
  (18, 18)	2.0
  (18, 19)	-1.0
  (19, 18)	-1.0
  (19, 19)	2.0
H=
   (0, 0)	2.0
  (0, 1)	-1.0
  (1, 0)	-1.0
  (1, 1)	2.0
  (1, 2)	-1.0
  (2, 1)	-1.0
  (2, 2)	2.0
  (2, 3)	-1.0
  (3, 2)	-1.0
  (3, 3)	2.0
  (3, 4)	-1.0
  (4, 3)	-1.0
  (4, 4)	2.0
  (4, 5)	-1.0
  (5, 4)	-1.0
  (5, 5)	2.0
  (5, 6)	-1.0
  (6, 5)	-1.0
  (6, 6)

In [15]:
N = 100

main_diag = 2*np.ones(N)
off_diag  = -1*np.ones(N-1)

H1 = diags(
    [off_diag, main_diag, off_diag],
    offsets=[-1,0,1],
    format='csr'
)

H2 = diags(
    [main_diag, off_diag, off_diag],
    offsets=[0,1,-1],
    format='coo'
)

print("H1 =\n", H1)
print()

print("H2 =\n", H2)
print()

print("H1 == H2 :", (H1 != H2).nnz == 0)

H1 =
   (0, 0)	2.0
  (0, 1)	-1.0
  (1, 0)	-1.0
  (1, 1)	2.0
  (1, 2)	-1.0
  (2, 1)	-1.0
  (2, 2)	2.0
  (2, 3)	-1.0
  (3, 2)	-1.0
  (3, 3)	2.0
  (3, 4)	-1.0
  (4, 3)	-1.0
  (4, 4)	2.0
  (4, 5)	-1.0
  (5, 4)	-1.0
  (5, 5)	2.0
  (5, 6)	-1.0
  (6, 5)	-1.0
  (6, 6)	2.0
  (6, 7)	-1.0
  (7, 6)	-1.0
  (7, 7)	2.0
  (7, 8)	-1.0
  (8, 7)	-1.0
  (8, 8)	2.0
  :	:
  (91, 91)	2.0
  (91, 92)	-1.0
  (92, 91)	-1.0
  (92, 92)	2.0
  (92, 93)	-1.0
  (93, 92)	-1.0
  (93, 93)	2.0
  (93, 94)	-1.0
  (94, 93)	-1.0
  (94, 94)	2.0
  (94, 95)	-1.0
  (95, 94)	-1.0
  (95, 95)	2.0
  (95, 96)	-1.0
  (96, 95)	-1.0
  (96, 96)	2.0
  (96, 97)	-1.0
  (97, 96)	-1.0
  (97, 97)	2.0
  (97, 98)	-1.0
  (98, 97)	-1.0
  (98, 98)	2.0
  (98, 99)	-1.0
  (99, 98)	-1.0
  (99, 99)	2.0

H2 =
   (0, 0)	2.0
  (1, 1)	2.0
  (2, 2)	2.0
  (3, 3)	2.0
  (4, 4)	2.0
  (5, 5)	2.0
  (6, 6)	2.0
  (7, 7)	2.0
  (8, 8)	2.0
  (9, 9)	2.0
  (10, 10)	2.0
  (11, 11)	2.0
  (12, 12)	2.0
  (13, 13)	2.0
  (14, 14)	2.0
  (15, 15)	2.0
  (16, 16)	2.0
  (17, 17)	2.0

# Sparse Matrix Formats in SciPy

## 1. COO Format (`"coo"`)

**COO = Coordinate Format**

Stores only non-zero elements.

### Best for:
- Building sparse matrices
- Adding elements easily

## 2. CSR Format (`"csr"`)

**CSR = Compressed Sparse Row**

Stores non-zero elements row-by-row.

### Best for:
- Fast matrix computations
- Matrix-vector multiplication
- Quantum Hamiltonians

### Advantage:
Efficient for numerical simulations.

## 3. CSC Format (`"csc"`)

**CSC = Compressed Sparse Column**

Stores non-zero elements column-by-column.

### Best for:
- Column operations
- Sparse matrix factorizations

### Advantage:
Efficient for column slicing.


## 4. DIA Format (`"dia"`)

**DIA = Diagonal Format**

Stores only the diagonals of a matrix.

### Best for:
- Tridiagonal matrices
- Finite difference Hamiltonian

Only the diagonals are stored.

# Summary Table

| Format | Meaning | Best Use |
|---|---|---|
| `"coo"` | Coordinate format | Building matrices |
| `"csr"` | Compressed Sparse Row | Fast computations |
| `"csc"` | Compressed Sparse Column | Column operations |
| `"dia"` | Diagonal storage | Banded/tridiagonal matrices |

## Dense Matrix 
\[O(N^2)\]

## Sparse Tridiagonal Matrix
\[N^2\] to \[O(N)\]


| Matrix Type | Stored Elements | Memory Complexity |
|---|---|---|
| Dense Matrix | \(N^2\) | \(O(N^2)\) |
| Sparse Tridiagonal Matrix | \(\sim 3N\) | \(O(N)\) |

***In sparse diagonal matrix construction, the main diagonal always corresponds to (offset 0) , upper diagonal (offset 1) and lower diagonal (offset −1) The values on both off-diagonals happen to be −1***

The neighboring diagonals are identified by their relative positions:

| Offset | Diagonal |
|---|---|
| \(0\) | Main diagonal |
| \(1\) | Upper diagonal |
| \(-1\) | Lower diagonal |


## SciPy: `dia`

⚠️ Don't use sparse matrices as input to NumPy functions!

There are very efficient methods for finding a few eigenpairs of a sparse matrix, such as the **Lanczos algorithm**.

# 2.4 Krylov Subspace Methods

Krylov subspace methods are numerical algorithms used to solve very large matrix problems efficiently without diagonalizing the full matrix.


### 2.4 The Lanczos algorithm

**Idea:** Find the largest / smallest eigenvalues of a very large (sparse) matrix $A = A^\dagger$ by projecting it onto a smaller subspace, the **Krylov subspace**:

$$\mathcal{K}_m = \text{span} \big\{ |v_1\rangle, A|v_1\rangle, A^2|v_1\rangle, \dots, A^{m-1}|v_1\rangle \big\} \quad \text{with } m \ll N$$

---

### Algorithmic Procedure:

Construct an orthonormal basis $\{|v_1\rangle, |v_2\rangle, \dots, |v_m\rangle\}$ of $\mathcal{K}_m$ step-by-step:

| Iterative Construction Steps | Mathematical Orthogonalization | Matrix Element Definition |
| :---: | :---: | :---: |
| **1. Choose initial state:** <br> Start with a random normed vector $|v_1\rangle$ | Set the boundary condition: <br><br> $$\beta_0 |v_0\rangle := 0$$ | Calculate diagonal elements: <br><br>$$\alpha_k = \langle v_k | A | v_k \rangle$$ |
| **2. Generate next vector:** <br> Apply $A$ and orthogonalize against previous vectors | Use the three-term recurrence relation: <br><br> $$|\tilde{v}_{k+1}\rangle = A|v_k\rangle - \alpha_k |v_k\rangle - \beta_{k-1}|v_{k-1}\rangle$$ | Calculate off-diagonal norms: <br><br>$$\beta_k = \big\| |\tilde{v}_{k+1}\rangle \big\|$$ |
| **3. Normalize:** <br> Obtain the next basis vector | Normalize the state vector: <br><br> $$|v_{k+1}\rangle = \frac{1}{\beta_k} |\tilde{v}_{k+1}\rangle$$ | *(Process repeats sequentially for $k = 1, 2, \dots, m$)* |

---

### Resulting Matrix Structure:

In this basis, the projection of $A$ onto $\mathcal{K}_m$ forms a symmetric **tridiagonal matrix** $T_m$:

$$T_m = \begin{pmatrix}
\alpha_1 & \beta_1 & 0 & \dots & 0 \\
\beta_1 & \alpha_2 & \beta_2 & \ddots & \vdots \\
0 & \beta_2 & \alpha_3 & \ddots & 0 \\
\vdots & \ddots & \ddots & \ddots & \beta_{m-1} \\
0 & \dots & 0 & \beta_{m-1} & \alpha_m
\end{pmatrix}$$

* The extreme eigenvalues of $T_m$ converge very rapidly to the extreme eigenvalues of $A$.
* **Computational Cost:** Since it only requires matrix-vector multiplications, the scaling is **$\mathcal{O}(m \cdot N)$** for sparse matrices!

## 2.4 Krylov subspace methods

### Motivation: Imaginary time evolution

$$e^{-H\tau} |\Psi_0\rangle = \sum_{k} e^{-E_k \tau} c_k |k\rangle$$

* $|\Psi_0\rangle$: some initial state (not normalized)
* $|k\rangle$: eigenstates
* $c_k = \langle k | \Psi_0 \rangle$
* $E_1 < E_2 < \dots$

$$= e^{-E_1 \tau} \Big( c_1 |1\rangle + \sum_{k=2}^{N} e^{-(E_k - E_1)\tau} c_k |k\rangle \Big)$$

* $|1\rangle$: ground state
* The summation term $\rightarrow 0$ for $\tau \rightarrow \infty$

$$\xrightarrow{\tau \rightarrow \infty} |1\rangle \quad \text{(if we keep the state normalized at any time)}$$

In principle one could use this method to find ground states by iteratively applying $e^{-H\tau} |\Psi_0\rangle$ with some fixed $\tau$ and renormalizing after each step. But how to calculate $e^{-H\tau}$, and also convergence may be slow...

Similarly, $H^k |\Psi_0\rangle$ (iteratively apply $H$) converges to the eigenvalue with largest magnitude:

$$H |\Psi_0\rangle = \sum_{k} c_k \lambda_k |k\rangle = \lambda_{\max} \sum_{k} c_k \frac{\lambda_k}{\lambda_{\max}} |k\rangle$$

* $\frac{\lambda_k}{\lambda_{\max}} < 1$ (in abs val.)
* $\rightarrow$ Portion of $|k_{\max}\rangle$ increases in each step, relative to other states.

"Matrix power method"

---

### Even more efficient: Lanczos algorithm

Build the "Krylov space" spanned by $\{|\Psi_0\rangle, H|\Psi_0\rangle, \dots, H^n|\Psi_0\rangle\} = \mathcal{K}_n(H_0)$ and represent $H$ in this subspace and diagonalize.

> **Note:** Lanczos algorithm is a special case of the Arnoldi iteration, for Hermitian matrices.

## Lanczos algorithm in detail:

Build an orthonormal basis of $\mathcal{K}_n(|\Psi_0\rangle)$ by Gram-Schmidt method and construct $h_{\mathcal{K}_n}$ on the fly:
$\{|q_1\rangle, \dots, |q_n\rangle\}$ ONB of $\mathcal{K}_n$

$$h = \begin{pmatrix} 
\langle q_1 | H | q_1 \rangle & \langle q_1 | H | q_2 \rangle & \langle q_1 | H | q_3 \rangle & \dots \\ 
\text{h.c. of} & \langle q_2 | H | q_2 \rangle & \langle q_2 | H | q_3 \rangle & \dots \\ 
\text{upper triangle} & \dots & \dots & \dots 
\end{pmatrix}$$

---

### 1st step:

* $|q_1\rangle = |\Psi_0\rangle$ (start)
* $|v_2\rangle = H |q_1\rangle$ (apply $H$)
* $|\tilde{v}_2\rangle = |v_2\rangle - |q_1\rangle \underbrace{\langle q_1 | v_2 \rangle}_{\text{1st matrix element we need: } h_{11} = \langle q_1 | H | q_1 \rangle}$ 

> **Gram-Schmidt:** subtract proj. on prev. found basis vectors

* $|q_2\rangle = \frac{|\tilde{v}_2\rangle}{\sqrt{\langle \tilde{v}_2 | \tilde{v}_2 \rangle}}$ (normalize)

---

### $n$-th step:

* $|v_n\rangle = H |q_{n-1}\rangle$
* $|\tilde{v}_n\rangle = |v_n\rangle - \sum_{k=1}^{n-1} |q_k\rangle \overbrace{\langle q_k | v_n \rangle}^{\langle q_k | H | q_{n-1} \rangle \text{ matrix element}}$

> **note:** $\langle q_k | v_n \rangle = 0$ for $k < n - 2$:
> 
> $\langle q_k | v_n \rangle = \langle q_k | H | q_{n-1} \rangle = \langle v_{k+1} | q_{n-1} \rangle$
> 
> $|q_{n-1}\rangle \perp |q_{k'}\rangle, \quad k' \le n-1$
> 
> $|v_{k+1}\rangle =$ superpos. of $|q_{k'}\rangle, \quad k' \le k+1$
> 
> $\Rightarrow \langle v_{k+1} | q_{n-1} \rangle = 0$ for $k+1 < n-1$

* $|q_n\rangle = \frac{|\tilde{v}_n\rangle}{\| |\tilde{v}_n\rangle \|}$

---

### Consequences:

1. Only need to subtract projection on **last 2 states**.
2. $h$ is **tri-diagonal**, eg. $\langle q_1 | H | q_3 \rangle = \langle q_1 | v_4 \rangle = 0$
   $\Rightarrow h$ can be diagonalized efficiently using **QR!**

## Properties of Lanczos:

* Only need to apply $H$ to states $\rightarrow$ scales well if $H$ is sparse ($\mathcal{O}(N)$).
* Not even needs $H$ explicitly but only a function that returns $H|\Psi\rangle$ upon input $|\Psi\rangle$.
* Converges fast to accurate approximation of largest magnitude eigenvalue(s).
  
  Convergence speed depends on $\frac{\lambda_2 - \lambda_1}{\lambda_N - \lambda_1}$, ratio between gap of largest to second largest eigenvalue and overall width of spectrum. Often $n \ll N$ iterations needed.

* To obtain states close to some energy $E^*$, e.g., ground states, use the **shift-invert technique**: Do Lanczos on $(H - E^*\mathbb{1})^{-1} =: G$.

  Converges to state just above $E^*$.

  * **apply $G$ to $|\Psi_k\rangle$**: solve $G^{-1} |\Psi_{k+1}\rangle = |\Psi_k\rangle$ for $|\Psi_{k+1}\rangle$ using e.g., Gauss elimination.

* Choice of initial state $|\Psi_0\rangle$ is important. Should have as large as possible ground state overlap. Typically Gaussian random vectors are used.